In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

from helper.config import load_config
import psycopg2

from_date = ''
to_date = ''

config = load_config()
with psycopg2.connect(**config) as conn:
    with conn.cursor() as cur:
            cur.execute(f"""
                            select to_timestamp(start_time+tz) AT TIME ZONE 'UTC' as start_time, to_timestamp(end_time+tz) AT TIME ZONE 'UTC' as end_time, app, device_id, device_model
                            from apple.fact_screentime 
                            --where date_trunc('day', to_timestamp(start_time+tz) AT TIME ZONE 'UTC') between to_date('{from_date}', 'yyyy-MM-dd') and to_date('{to_date}', 'yyyy-MM-dd')
                        """)
            df = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description]) 

# Sample DataFrame
# df = pd.read_csv("your_dataset.csv")

# Convert to datetime if not already
df['start_time'] = pd.to_datetime(df['start_time'])
df['end_time'] = pd.to_datetime(df['end_time'])

# Calculate time used (in minutes)
df['time_used'] = (df['end_time'] - df['start_time']).dt.total_seconds() / 60

# Extract date for filtering and highlighting
df['start_date'] = df['start_time'].dt.date

# 1. Time-Based Visualization with Weekend Highlight
time_per_day = df.groupby('start_date')['time_used'].sum().reset_index()
time_per_day['day_of_week'] = pd.to_datetime(time_per_day['start_date']).dt.day_name()
time_per_day['is_weekend'] = time_per_day['day_of_week'].isin(['Saturday', 'Sunday'])

fig = px.bar(time_per_day, x='start_date', y='time_used', color='is_weekend',
             color_discrete_map={True: 'red', False: 'blue'},
             title='Total App Usage Time per Day (Minutes) with Weekend Highlight')
fig.update_xaxes(title='Date')
fig.update_yaxes(title='Total Time Used (Minutes)')
fig.show()

# 2. App Time Used with Day Filter
date_filter = widgets.Dropdown(
    options=[str(date) for date in time_per_day['start_date']],
    value=str(time_per_day['start_date'].max()),
    description='Select Date:'
)

output_gantt = widgets.Output()
output_app = widgets.Output()
output_device = widgets.Output()

def update_gantt_chart(selected_date):
    with output_gantt:
        clear_output(wait=True)
        filtered_df = df[df['start_date'] == pd.to_datetime(selected_date).date()].copy()
        filtered_df['device_model'] = filtered_df['device_model'].fillna('Mac Mini')
        app_usage = filtered_df.groupby('app')['time_used'].sum().reset_index().sort_values(by='time_used', ascending=False)
        top_20_apps = app_usage.head(20)['app'].tolist()
        filtered_df = filtered_df[filtered_df['app'].isin(top_20_apps)]

        fig = px.timeline(filtered_df, x_start='start_time', x_end='end_time', y='app', color='device_model',
                          title=f'App Usage Timeline per App on {selected_date}')
        fig.update_yaxes(categoryorder='total ascending')
        fig.update_layout(height=600)
        fig.show()

def update_app_usage(selected_date):
    with output_app:
        clear_output(wait=True)
        filtered_df = df[df['start_date'] == pd.to_datetime(selected_date).date()]
        app_usage = filtered_df.groupby('app')['time_used'].sum().reset_index().sort_values(by='time_used', ascending=False)
        app_usage = app_usage[app_usage['time_used'] > 0]

        if len(app_usage) > 20:
            app_usage = app_usage.head(20)

        fig = px.bar(app_usage, x='time_used', y='app', orientation='h',
                     title=f'Total Time Used by App (Minutes) on {selected_date}')
        fig.update_xaxes(title='Total Time Used (Minutes)')
        fig.update_layout(height=600)
        fig.show()

def update_device_usage(selected_date):
    with output_device:
        clear_output(wait=True)
        filtered_df = df[df['start_date'] == pd.to_datetime(selected_date).date()].copy()
        filtered_df['device_model'] = filtered_df['device_model'].fillna('Mac Mini')
        device_usage = filtered_df.groupby('device_model')['time_used'].sum().reset_index().sort_values(by='time_used', ascending=False)

        fig = px.bar(device_usage, x='time_used', y='device_model', orientation='h',
                     title=f'Total Time Used by Device Model (Minutes) on {selected_date}')
        fig.update_xaxes(title='Total Time Used (Minutes)')
        fig.show()

def on_date_change(change):
    selected_date = change['new']
    update_gantt_chart(selected_date)
    update_app_usage(selected_date)
    update_device_usage(selected_date)

date_filter.observe(on_date_change, names='value')

display(date_filter, output_gantt, output_app, output_device)

# Initial render
update_gantt_chart(date_filter.value)
update_app_usage(date_filter.value)
update_device_usage(date_filter.value)

Dropdown(description='Select Date:', index=44, options=('2025-01-14', '2025-01-15', '2025-01-16', '2025-01-17'…

Output()

Output()

Output()